# CENG562 Term Project

Bare-bones Colab notebook for BM25 and dense retrieval on the TREC Clinical Trials collection.

This notebook keeps the structure close to the original Colab workflow while limiting the scope to:
- BM25 retrieval with PyTerrier/Terrier
- dense retrieval with SentenceTransformers + FAISS
- optional dense embedding caching to Google Drive


In [ ]:
# Optional: mount Google Drive for persistent embedding cache
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip -q install ir_datasets nltk tqdm
!pip install -q pyterrier
!pip install -q "pyterrier[java]"
!pip install -q sentence-transformers faiss-cpu

## Imports and Settings

In [ ]:
import os
import re
from collections import defaultdict

import faiss
import ir_datasets
import numpy as np
import pandas as pd
import pyterrier as pt
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

os.environ["IR_DATASETS_HOME"] = "/content/ir_datasets"

DOC_DATASET_ID = "clinicaltrials/2021"
QUERY_DATASET_ID = "clinicaltrials/2021/trec-ct-2021"

TEST_MAX_DOCS = 50000   # set to None for the full collection
FINAL_MAX_DOCS = None
TOP_K = 1000
DENSE_BATCH_SIZE = 64
DENSE_MODEL_NAME = "NeuML/pubmedbert-base-embeddings"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if not pt.java.started():
    pt.java.init()

## Document Processing

In [ ]:
def build_document_text(doc):
    title = getattr(doc, "title", "") or ""
    condition = getattr(doc, "condition", "") or ""
    summary = getattr(doc, "summary", "") or ""
    detailed_description = getattr(doc, "detailed_description", "") or ""
    eligibility = getattr(doc, "eligibility", "") or ""

    joined = " ".join([
        title,
        condition,
        summary,
        detailed_description,
        eligibility,
    ]).strip()

    raw_doc = {
        "title": title,
        "condition": condition,
        "summary": summary,
        "detailed_description": detailed_description,
        "eligibility": eligibility,
    }
    return joined, raw_doc


def build_dense_document_text(doc):
    title = getattr(doc, "title", "") or ""
    condition = getattr(doc, "condition", "") or ""
    summary = getattr(doc, "summary", "") or ""
    eligibility = getattr(doc, "eligibility", "") or ""
    return " ".join([title, condition, summary, eligibility]).strip()

In [ ]:
def get_docs_df(max_docs=None):
    doc_dataset = ir_datasets.load(DOC_DATASET_ID)
    total_docs = max_docs if max_docs is not None else doc_dataset.docs_count()

    rows = []
    raw_docs = {}

    for i, doc in enumerate(tqdm(doc_dataset.docs_iter(), total=total_docs, desc="Preparing BM25 docs")):
        if max_docs is not None and i >= max_docs:
            break

        doc_id = doc.doc_id
        full_text, raw_doc = build_document_text(doc)

        if not full_text.strip():
            continue

        rows.append({"docno": doc_id, "text": full_text})
        raw_docs[doc_id] = raw_doc

    return pd.DataFrame(rows), raw_docs


def get_dense_docs_df(max_docs=None):
    doc_dataset = ir_datasets.load(DOC_DATASET_ID)
    total_docs = max_docs if max_docs is not None else doc_dataset.docs_count()

    rows = []
    for i, doc in enumerate(tqdm(doc_dataset.docs_iter(), total=total_docs, desc="Preparing dense docs")):
        if max_docs is not None and i >= max_docs:
            break

        dense_text = build_dense_document_text(doc)
        if not dense_text.strip():
            continue

        rows.append({"docno": doc.doc_id, "text": dense_text})

    return pd.DataFrame(rows)

In [ ]:
docs_df, raw_docs = get_docs_df(FINAL_MAX_DOCS)
print("Prepared BM25 docs:", len(docs_df))

## BM25 Indexing and Retrieval

In [ ]:
index_dir = "/content/clinical_trials_pt_index"
indexer = pt.DFIndexer(index_dir, overwrite=True)
index_ref = indexer.index(docs_df["text"], docs_df["docno"])
print("Index built at:", index_dir)

In [ ]:
query_dataset = ir_datasets.load(QUERY_DATASET_ID)
queries_df = pd.DataFrame([
    {"qid": q.query_id, "query": q.text}
    for q in query_dataset.queries_iter()
])

qrels = defaultdict(dict)
for qr in query_dataset.qrels_iter():
    qrels[qr.query_id][qr.doc_id] = qr.relevance

print("Loaded queries:", len(queries_df))
print("Loaded qrels for queries:", len(qrels))

In [ ]:
def normalize_query_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_bm25_query(text: str) -> str:
    return normalize_query_text(text)

def build_dense_query(text: str) -> str:
    return normalize_query_text(text)

In [ ]:
bm25 = pt.BatchRetrieve(
    index_ref,
    wmodel="BM25",
    controls={"bm25.k_1": 1.5, "bm25.b": 0.75},
    num_results=TOP_K,
)

bm25_queries_df = queries_df.copy()
bm25_queries_df["query"] = bm25_queries_df["query"].apply(build_bm25_query)

bm25_results_df = bm25.transform(bm25_queries_df)
bm25_results_df = bm25_results_df.sort_values(["qid", "rank"]).reset_index(drop=True)
bm25_results_df.head()

## Dense Retrieval

In [ ]:
dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=device)
print("Dense model loaded:", DENSE_MODEL_NAME)

In [ ]:
dense_doc_df = get_dense_docs_df(FINAL_MAX_DOCS)
print("Prepared dense docs:", len(dense_doc_df))

if os.path.exists("/content/drive/MyDrive"):
    EMB_PATH = "/content/drive/MyDrive/dense_doc_embeddings.npy"
    DOCNO_PATH = "/content/drive/MyDrive/dense_docnos.npy"
    print("Google Drive is mounted. Saving embeddings to Drive.")
else:
    EMB_PATH = "/content/dense_doc_embeddings.npy"
    DOCNO_PATH = "/content/dense_docnos.npy"
    print("Google Drive is not mounted. Saving embeddings to local Colab storage.")

if os.path.exists(EMB_PATH) and os.path.exists(DOCNO_PATH):
    dense_doc_embeddings = np.load(EMB_PATH)
    dense_docnos = np.load(DOCNO_PATH, allow_pickle=True).tolist()
    print("Loaded cached dense document embeddings.")
else:
    dense_doc_embeddings = dense_model.encode(
        dense_doc_df["text"].tolist(),
        batch_size=DENSE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    dense_docnos = dense_doc_df["docno"].tolist()
    np.save(EMB_PATH, dense_doc_embeddings)
    np.save(DOCNO_PATH, np.array(dense_docnos, dtype=object))
    print("Computed and saved dense document embeddings.")

print("Dense doc embeddings shape:", dense_doc_embeddings.shape)

In [ ]:
dense_dim = dense_doc_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dense_dim)
faiss_index.add(dense_doc_embeddings)
print("FAISS index size:", faiss_index.ntotal)

In [ ]:
dense_queries_df = queries_df.copy()
dense_queries_df["query"] = dense_queries_df["query"].apply(build_dense_query)

dense_query_embeddings = dense_model.encode(
    dense_queries_df["query"].tolist(),
    batch_size=DENSE_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

scores, indices = faiss_index.search(dense_query_embeddings, TOP_K)

dense_rows = []
for q_idx, qid in enumerate(dense_queries_df["qid"].tolist()):
    for rank_idx, (doc_idx, score) in enumerate(zip(indices[q_idx], scores[q_idx]), start=1):
        dense_rows.append({
            "qid": qid,
            "docno": dense_docnos[doc_idx],
            "score": float(score),
            "rank": rank_idx,
        })

dense_results_df = pd.DataFrame(dense_rows)
dense_results_df = dense_results_df.sort_values(["qid", "rank"]).reset_index(drop=True)
dense_results_df.head()

## Quick qualitative check

In [ ]:
sample_qid = queries_df.iloc[0]["qid"]
sample_query = queries_df.iloc[0]["query"]

sample_bm25 = bm25_results_df[bm25_results_df["qid"] == sample_qid].sort_values("rank").head(5)
sample_dense = dense_results_df[dense_results_df["qid"] == sample_qid].sort_values("rank").head(5)

print("QUERY:")
print(sample_query[:1000])

print("\nTOP BM25 RESULTS:\n")
for row in sample_bm25.itertuples(index=False):
    raw_doc = raw_docs.get(row.docno, {})
    print(f"{int(row.rank)}. {row.docno} | score={row.score:.4f}")
    print(f"   title      : {raw_doc.get('title', '')}")
    print(f"   condition  : {raw_doc.get('condition', '')}")
    print()

print("\nTOP DENSE RESULTS:\n")
for row in sample_dense.itertuples(index=False):
    raw_doc = raw_docs.get(row.docno, {})
    print(f"{int(row.rank)}. {row.docno} | score={row.score:.4f}")
    print(f"   title      : {raw_doc.get('title', '')}")
    print(f"   condition  : {raw_doc.get('condition', '')}")
    print()

In [ ]:
def qrels_at_threshold(qrels, min_rel=1):
    thresholded = {}
    for qid, docs in qrels.items():
        thresholded[qid] = {docno: 1 if rel >= min_rel else 0 for docno, rel in docs.items()}
    return thresholded


def evaluate_run(results_df, qrels_binary, k=1000):
    grouped_results = {
        qid: group.sort_values("rank")[["docno", "score"]].to_dict("records")
        for qid, group in results_df.groupby("qid")
    }

    precisions, recalls, maps, mrrs, ndcgs = [], [], [], [], []

    for qid, rel_docs in qrels_binary.items():
        retrieved = grouped_results.get(qid, [])[:k]
        relevant_set = {docno for docno, rel in rel_docs.items() if rel > 0}
        total_relevant = len(relevant_set)

        hits = 0
        ap_sum = 0.0
        rr = 0.0
        dcg = 0.0

        for idx, row in enumerate(retrieved, start=1):
            gain = 1 if row["docno"] in relevant_set else 0
            if gain:
                hits += 1
                ap_sum += hits / idx
                if rr == 0.0:
                    rr = 1.0 / idx
            dcg += gain / np.log2(idx + 1)

        ideal_hits = min(total_relevant, k)
        idcg = sum(1.0 / np.log2(i + 1) for i in range(2, ideal_hits + 2))

        precisions.append(hits / k if k else 0.0)
        recalls.append(hits / total_relevant if total_relevant else 0.0)
        maps.append(ap_sum / total_relevant if total_relevant else 0.0)
        mrrs.append(rr)
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return {
        f"P@{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"R@{k}": float(np.mean(recalls)) if recalls else 0.0,
        "MAP": float(np.mean(maps)) if maps else 0.0,
        "MRR": float(np.mean(mrrs)) if mrrs else 0.0,
        f"nDCG@{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
    }


thresholds = [1, 2]
comparison_rows = []

for rel_threshold in thresholds:
    qrels_binary = qrels_at_threshold(qrels, min_rel=rel_threshold)

    bm25_metrics = evaluate_run(bm25_results_df, qrels_binary, k=TOP_K)
    dense_metrics = evaluate_run(dense_results_df, qrels_binary, k=TOP_K)

    for system_name, metrics in [("BM25", bm25_metrics), ("Dense", dense_metrics)]:
        row = {"system": system_name, "threshold": f"rel>={rel_threshold}"}
        row.update(metrics)
        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df